# XGBoost — Pipeline Tổng Hợp: Single / Nhánh Sông / Lưu Vực / Fine-tune × 3 Mùa

72 booster (24 horizon x 3 quantile P10/P50/P90) mỗi nhóm. Không cần GPU.
Xem chi tiết từng Pha ở docstring đầu file generator
(`kaggle/generate_notebook_master.py`).


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "xgboost>=2.0", "huggingface_hub"])


In [ ]:
import os, json
import numpy as np
import pandas as pd
import xgboost as xgb

SKIP_IF_DONE = True  # False = train lại từ đầu, bỏ qua toàn bộ resume-check
SEASONS = ["all", "dry", "rainy"]


## config/reservoirs.py + config/settings.py

In [ ]:
# config/reservoirs.py
# Copy nguyen tu LSTM_Py_Backend_v2/config/reservoirs.py -- 16 ho Quang Nam/Da Nang.
# "idx" dung lam chi so one-hot reservoir cho model GLOBAL (RF dung chung 1 bo
# Quantile Regression Forest cho ca 16 ho, khac LSTM_Py_Backend_v2 train rieng
# tung ho -- xem README.md).

RESERVOIRS = {
    1: {"idx": 0, "name": "HO A VUONG", "lat": 15.815, "lon": 107.63, "river_basin": "Vu Gia"},
    2: {"idx": 1, "name": "HO DAK MI 4", "lat": 15.45285, "lon": 107.83250, "river_basin": "Vu Gia"},
    3: {"idx": 2, "name": "HO SONG BUNG 4", "lat": 15.726, "lon": 107.637, "river_basin": "Vu Gia"},
    4: {"idx": 3, "name": "HO SONG TRANH 2", "lat": 15.326, "lon": 108.125, "river_basin": "Thu Bồn"},
    7: {"idx": 4, "name": "HO SONG BUNG 4A", "lat": 15.765, "lon": 107.679, "river_basin": "Vu Gia"},
    8: {"idx": 5, "name": "HO SONG BUNG 5", "lat": 15.808, "lon": 107.7473, "river_basin": "Vu Gia"},
    9: {"idx": 6, "name": "HO SONG BUNG 2", "lat": 15.7145, "lon": 107.3970, "river_basin": "Vu Gia"},
    11: {"idx": 7, "name": "HO SONG BUNG 6", "lat": 15.82, "lon": 107.78, "river_basin": "Vu Gia"},
    12: {"idx": 8, "name": "HO SONG TRANH 3", "lat": 15.4445, "lon": 108.1430, "river_basin": "Thu Bồn"},
    13: {"idx": 9, "name": "HO ZA HUNG", "lat": 15.86005, "lon": 107.654, "river_basin": "Vu Gia"},
    14: {"idx": 10, "name": "HO DAK MI 3", "lat": 15.33, "lon": 107.81, "river_basin": "Vu Gia"},
    15: {"idx": 11, "name": "HO KHE DIEN", "lat": 15.71279, "lon": 107.92872, "river_basin": "Thu Bồn"},
    16: {"idx": 12, "name": "HO SONG CON 2", "lat": 15.90558, "lon": 107.8234, "river_basin": "Vu Gia"},
    17: {"idx": 13, "name": "HO SONG TRANH 4", "lat": 15.53666, "lon": 108.152, "river_basin": "Thu Bồn"},
    18: {"idx": 14, "name": "HO DAK MI 2", "lat": 15.23832, "lon": 107.8100, "river_basin": "Vu Gia"},
    19: {"idx": 15, "name": "HO DAK MI 4C", "lat": 15.4643, "lon": 107.92893, "river_basin": "Thu Bồn"},
}
NUM_RESERVOIRS = len(RESERVOIRS)

# ── 1. PHÂN CHIA THEO 4 NHÁNH SÔNG (SUB-BASINS) ──────────────────────────────
RIVER_BRANCHES = {
    "A_VUONG": [1, 13],
    "SONG_BUNG": [9, 3, 7, 8, 11],
    "DAK_MI": [18, 14, 2, 19],
    "SONG_TRANH": [4, 12, 17, 15],  # Kèm Khe Diên theo yêu cầu kịch bản
}

# ── 2. BIẾN THỂ THỬ NGHIỆM CHO HỒ SÔNG CÔN 2 (RID=16) ──────────────────────
SONG_CON_2_VARIANTS = {
    "A_VUONG": [1, 13, 16],
    "SONG_BUNG": [9, 3, 7, 8, 11, 16],
}

# ── 3. PHÂN CHIA THEO LƯU VỰC SÔNG (MAIN BASINS) ─────────────────────────────
# Kịch bản thử nghiệm theo yêu cầu người dùng:
RIVER_BASINS_EXPERIMENT = {
    "Vu Gia": [4, 12, 17, 15],
    "Thu Bồn": [1, 13, 9, 3, 7, 8, 11, 18, 14, 2, 19, 16],
}

# Phân chia theo chuẩn thủy văn tự nhiên:
RIVER_BASINS_NATURAL = {
    "Vu Gia": [1, 2, 3, 7, 8, 9, 11, 13, 14, 16, 18],
    "Thu Bồn": [4, 12, 15, 17, 19],
}

RIVER_BASINS = RIVER_BASINS_EXPERIMENT

# ── 4. PHÂN CHIA THEO MÙA (SEASONAL SPLITTING) ──────────────────────────────
SEASON_MONTHS = {
    "dry": [1, 2, 3, 4, 5, 6, 7, 8],
    "rainy": [9, 10, 11, 12],
    "all": list(range(1, 13)),
}


def get_reservoirs_by_branch(branch_name: str) -> dict:
    """Trả về dict chứa các hồ thuộc nhánh sông chỉ định."""
    rids = RIVER_BRANCHES.get(branch_name.upper(), [])
    return {rid: RESERVOIRS[rid] for rid in rids if rid in RESERVOIRS}


def get_reservoirs_by_basin(basin_name: str, use_natural: bool = False) -> dict:
    """Trả về dict chứa các hồ thuộc lưu vực sông chỉ định."""
    basins = RIVER_BASINS_NATURAL if use_natural else RIVER_BASINS_EXPERIMENT
    rids = basins.get(basin_name, [])
    return {rid: RESERVOIRS[rid] for rid in rids if rid in RESERVOIRS}


In [ ]:
# config/settings.py
HORIZON = 24                    # 24h du bao (giong LSTM/XGBoost)
QUANTILES = [0.1, 0.5, 0.9]     # P10/P50/P90 -- dung contract voi ForecastRF (Node)

# Split 60% train / 20% val / 20% test THEO THOI GIAN (data/tabular_dataset.py
# ::split_60_20_20) -- theo yeu cau phuong phap cua du an, thay cho fixed-date
# split truoc day.

# Trong so mua lu Vu Gia - Thu Bon (thang 9 -> thang 1 nam sau) khi train --
# xem data/tabular_dataset.py::RAINY_SEASON_MONTHS. Nhan them vao trong so
# theo bien do dinh lu da co (top5%->2x, top1%->3x).
RAINY_SEASON_WEIGHT = 1.5

# Nguon backup Hugging Face khi khong co data local/Kaggle input -- xem
# LSTM_Py_Backend_v2/kaggle/generate_notebook_all.py (cung 1 nguon). LUU Y:
# repo nay hien TRONG (khong co file zip that) -- xem README.md.
HF_REPO_ID = "Anvo2004/dataset_all_lake"
HF_ZIP_FILENAME = "datasets_all_reservoirs.zip"


## data/tabular_dataset.py

In [ ]:
# data/tabular_dataset.py
"""
Doc du lieu tabular tu dataset LSTM_Py_Backend_v2 da build san (sliding-window
hindcast, xem LSTM_Py_Backend_v2/data/dataset_builder.py) -- KHONG con phu
thuoc Data_Tung_Ho_Ma_Tran_Rong/ (Excel goc) truc tiep.

Feature vector = dong CUOI CUNG cua hindcast window (= "hien tai", da chua
day du lag/rolling feature nen cua 240h qua khu) + oracle mua/khi tuong
tuong lai (X_nwp buoc dau tien) + one-hot reservoir.

Oracle rain (theo yeu cau phuong phap cua du an): "mua du bao" dua vao model
LA du lieu mua THUC TE da xay ra trong khoang tuong lai (khong phai du bao
that) -- model hoc quy luat mua->lu tu du lieu hoan chinh. Luc serving (xem
main_api.py), phan nay duoc thay bang du bao Open-Meteo that (khong con la
oracle nua, co sai so du bao thuc te) -- day la cach lam CHU DICH, giong het
LSTM_Py_Backend/v2 dang dung (X_future/X_nwp), khong phai loi train/serve
mismatch nhu ghi chu cu cua file nay tung noi.

Tim nguon du lieu theo thu tu uu tien:
  1. Local dev: ../LSTM_Py_Backend_v2/datasets/<Ten_Ho>/v2_*.npy
  2. Kaggle: /kaggle/input/**/v2_X_hindcast.npy (dataset da attach vao notebook)
  3. Hugging Face Hub: Anvo2004/dataset_all_lake (LUU Y: repo nay hien TRONG,
     khong co file zip that -- xem README.md. Fallback nay se loi neu ca local
     lan Kaggle input deu khong co, phai tu Add Input Dataset tren Kaggle.)
"""
import os
import numpy as np




_LOCAL_CANDIDATES = [
    os.path.join("..", "LSTM_Py_Backend_v2", "datasets"),
    os.path.join("LSTM_Py_Backend_v2", "datasets"),
]

# Danh sach 47 feature hindcast (thu tu CHINH XAC khop cot cuoi cung cua
# v2_X_hindcast.npy -- xem LSTM_Py_Backend_v2/data/dataset_builder.py::FEATURES).
FEATURES = [
    "rain", "rain_3h", "rain_6h", "rain_12h", "rain_24h",
    "rain_48h", "rain_72h", "rain_96h", "rain_120h", "rain_168h",
    "rain_intensity", "rain_12h_std", "rain_24h_max",
    "rain_lag_1", "rain_lag_3", "rain_lag_6", "rain_lag_12", "rain_lag_24",
    "inflow", "inflow_prev", "inflow_diff", "inflow_diff_2",
    "inflow_3h_avg", "inflow_6h_avg", "inflow_12h_avg",
    "inflow_24h_avg", "inflow_48h_avg", "inflow_rising",
    "rain_inflow_interaction", "soil_moisture_x_inflow",
    "water_level", "outflow", "Z_diff", "Z_24h_avg", "outflow_diff", "Q_ratio",
    "temperature", "relative_humidity", "pressure", "et0", "wind_speed",
    "hour_sin", "hour_cos", "doy_sin", "doy_cos", "month_sin", "month_cos",
]  # 47 features

# 6 feature "tuong lai" (oracle luc train, du bao Open-Meteo luc serving) --
# lay dung buoc dau tien (t+1h) cua v2_X_nwp.npy, xem
# LSTM_Py_Backend_v2/data/dataset_builder.py::_build_nwp_window() /
# NWP_FEATURES. rain_fc_24h la tich luy mua tu t+1 den t+24 (ca cua so du
# bao), nen dung chung cho ca 24 model horizon (h+1..h+24) khong sai logic --
# giong cach LSTM dung 1 X_future cho ca 24 buoc output.
FUTURE_FEATURES = ["rain_fc", "rain_fc_3h", "rain_fc_6h", "rain_fc_24h", "temp_fc", "wind_fc"]

ALL_FEATURES = FEATURES + FUTURE_FEATURES  # 53 feature co so (chua tinh one-hot ho)


def _find_dataset_dirs() -> dict:
    """Tra ve {reservoir_key: folder_path chua v2_X_hindcast.npy}."""
    for local_root in _LOCAL_CANDIDATES:
        if os.path.isdir(local_root):
            found = {}
            for name in os.listdir(local_root):
                p = os.path.join(local_root, name)
                if os.path.exists(os.path.join(p, "v2_X_hindcast.npy")):
                    found[name] = p
            if found:
                print(f"[data] Dung nguon local: {local_root}/ ({len(found)} ho)")
                return found

    kaggle_root = "/kaggle/input"
    if os.path.isdir(kaggle_root):
        found = {}
        for root, _, files in os.walk(kaggle_root):
            if "v2_X_hindcast.npy" in files:
                found[os.path.basename(root)] = root
        if found:
            print(f"[data] Dung nguon Kaggle input: {kaggle_root} ({len(found)} ho)")
            return found

    print(f"[data] Khong tim thay data local/Kaggle -> tai tu Hugging Face '{HF_REPO_ID}'...")
    from huggingface_hub import hf_hub_download
    import zipfile
    zip_path = hf_hub_download(repo_id=HF_REPO_ID, filename=HF_ZIP_FILENAME, repo_type="dataset")
    extract_dir = "/kaggle/working/datasets" if os.path.isdir("/kaggle/working") else "./_hf_datasets"
    os.makedirs(extract_dir, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_dir)
    found = {}
    for root, _, files in os.walk(extract_dir):
        if "v2_X_hindcast.npy" in files:
            found[os.path.basename(root)] = root
    if not found:
        raise FileNotFoundError(
            "Khong tim thay v2_X_hindcast.npy o local, Kaggle input, lan Hugging Face "
            "(repo Anvo2004/dataset_all_lake hien khong co file zip that -- phai tu Add "
            "Input Dataset tren Kaggle, xem README.md)."
        )
    print(f"[data] Da tai va giai nen tu Hugging Face ({len(found)} ho)")
    return found


def build_tabular_dataset():
    """
    Tra ve:
      X   (N, 47 hindcast + 6 oracle-future + NUM_RESERVOIRS one-hot) float32
      y   (N, HORIZON)                                                 float32, sqrt-space
      rid (N,)                                                         int64, reservoir idx (0..15)
      ts  (N,)                                                         datetime64[s]
    """
    dirs = _find_dataset_dirs()
    key_to_idx = {info["name"].replace(" ", "_"): info["idx"] for _, info in RESERVOIRS.items()}

    X_list, y_list, rid_list, ts_list = [], [], [], []
    for key, path in sorted(dirs.items()):
        if key not in key_to_idx:
            print(f"  [SKIP] {key}: khong co trong config/reservoirs.py")
            continue
        idx = key_to_idx[key]
        X_hind = np.load(os.path.join(path, "v2_X_hindcast.npy"), mmap_mode="r")
        X_nwp = np.load(os.path.join(path, "v2_X_nwp.npy"), mmap_mode="r")
        y = np.load(os.path.join(path, "v2_y.npy"))
        ts = np.load(os.path.join(path, "v2_timestamps.npy"))

        X_last = np.asarray(X_hind[:, -1, :], dtype=np.float32)   # dong cuoi hindcast = "hien tai"
        X_future = np.asarray(X_nwp[:, 0, :], dtype=np.float32)   # buoc dau tien cua cua so du bao (t+1h)
        onehot = np.zeros((len(X_last), NUM_RESERVOIRS), dtype=np.float32)
        onehot[:, idx] = 1.0

        X_list.append(np.concatenate([X_last, X_future, onehot], axis=1))
        y_list.append(np.asarray(y, dtype=np.float32))
        rid_list.append(np.full(len(X_last), idx, dtype=np.int64))
        ts_list.append(ts)
        print(f"  [{key}] {len(X_last):,} samples")

    if not X_list:
        raise RuntimeError("Khong build duoc mau nao -- kiem tra lai thu muc dataset.")

    X = np.concatenate(X_list, axis=0)
    y = np.concatenate(y_list, axis=0)
    rid = np.concatenate(rid_list, axis=0)
    ts = np.concatenate(ts_list, axis=0)
    print(f"Total: {len(X):,} samples | X={X.shape} (47 hindcast + 6 oracle-future + "
          f"{NUM_RESERVOIRS} one-hot) | y={y.shape}")
    return X, y, rid, ts


def split_60_20_20(ts: np.ndarray):
    """
    Split ~79% train / ~8% validation / ~13% test THEO THOI GIAN (chronological,
    khong phai random) -- tranh data leakage (khong de mau tuong lai lot vao
    tap train khi mau qua khu nam trong test).

    Ten ham giu nguyen "60_20_20" cho khop cac noi da import/goi (train_xgb.py,
    evaluate_xgb.py...) nhung ty le that su da doi (yeu cau dung nhieu du lieu
    train hon) -- khop dung voi moc ngay co dinh ben LSTM_Py_Backend_v2
    (config/settings.py: train_end=2025-02-28 ~79%, val_end=2025-06-24 ~87%)
    de 2 backend so sanh tuong duong nhau. Tap test van du ~6 thang, bao gom ca
    mua kho (T6-8) lan mua mua (T9-12) -- khong rut xuong muc chi con 1 mua.

    Cutoff tinh theo % THOI GIAN da troi qua (khong phai % SO MAU), vi so mau
    khong deu tuyet doi giua cac thang (thang du/thang thieu ngay).
    """
    t_min, t_max = ts.min(), ts.max()
    span = (t_max - t_min).astype("timedelta64[s]").astype(np.int64)
    cutoff_60 = t_min + np.timedelta64(int(span * 0.7895), "s")
    cutoff_80 = t_min + np.timedelta64(int(span * 0.8696), "s")

    train_idx = np.where(ts < cutoff_60)[0]
    val_idx = np.where((ts >= cutoff_60) & (ts < cutoff_80))[0]
    test_idx = np.where(ts >= cutoff_80)[0]
    print(f"[split ~79/8/13] train <{cutoff_60} | val [{cutoff_60}, {cutoff_80}) | test >={cutoff_80}")
    return train_idx, val_idx, test_idx


# Mua lu Vu Gia - Thu Bon: thang 9 nam nay -> thang 1 nam sau (yeu cau uu tien
# bat dinh lu cua du an). Dung o training/train_rf.py, train_xgb.py de tang
# trong so mau trong mua lu, cong them trong so theo bien do dinh lu da co.
RAINY_SEASON_MONTHS = {9, 10, 11, 12, 1}


## training/event_metrics.py

In [ ]:
# training/event_metrics.py
"""
Copy nguyen tu LSTM_Py_Backend_v2/training/event_metrics.py -- logic khong
phu thuoc model/framework, dung lai y het cho RF.
"""

import numpy as np


def nse_single(obs: np.ndarray, pred: np.ndarray) -> float:
    """NSE (Nash-Sutcliffe Efficiency) trên 1 mảng 1D đã flatten."""
    ss_res = float(np.sum((obs - pred) ** 2))
    ss_tot = float(np.sum((obs - obs.mean()) ** 2))
    if ss_tot < 1e-8:
        return float("nan")
    return 1.0 - ss_res / ss_tot


def kge_single(obs: np.ndarray, pred: np.ndarray) -> float:
    """KGE (Kling-Gupta Efficiency) tren 1 mang 1D -- bo sung cho NSE de
    tach ro loi do tuong quan (r), do lech bien thien (alpha), do lech
    trung binh (beta)."""
    obs_mean, pred_mean = obs.mean(), pred.mean()
    obs_std, pred_std = obs.std(), pred.std()
    if obs_std < 1e-8 or abs(obs_mean) < 1e-8 or pred_std < 1e-8:
        return float("nan")
    r = float(np.corrcoef(obs, pred)[0, 1])
    if np.isnan(r):
        return float("nan")
    alpha = pred_std / obs_std
    beta = pred_mean / obs_mean
    return 1.0 - float(np.sqrt((r - 1) ** 2 + (alpha - 1) ** 2 + (beta - 1) ** 2))


def r2_single(obs: np.ndarray, pred: np.ndarray) -> float:
    """Hệ số xác định R2 (Coefficient of Determination) giữa obs và pred."""
    ss_res = float(np.sum((obs - pred) ** 2))
    ss_tot = float(np.sum((obs - obs.mean()) ** 2))
    if ss_tot < 1e-8:
        return float("nan")
    return float(1.0 - ss_res / ss_tot)


def extract_lead_time_series(
    preds: np.ndarray,   # (N, T) point forecast, đơn vị gốc (m3/s)
    obs: np.ndarray,     # (N, T)
    lead_idx: int,       # 0-based: 0 = giờ thứ 1, 23 = giờ thứ 24, ...
):
    """Trích chuỗi liên tục obs/pred tại 1 lead-time cố định. Giả định test
    set sliding-window KHÔNG shuffle (đúng với cách build_tabular_dataset())."""
    return obs[:, lead_idx].copy(), preds[:, lead_idx].copy()


def nse_per_horizon(
    preds: np.ndarray,   # (N, T) point forecast (đơn vị gốc, không phải sqrt)
    obs: np.ndarray,     # (N, T)
    group_hours: int = 6,
) -> list:
    """NSE riêng cho từng nhóm lead-time (mặc định 6h/nhóm cho horizon 24h)."""
    N, T = preds.shape
    n_groups = (T + group_hours - 1) // group_hours
    results = []
    for g in range(n_groups):
        lo, hi = g * group_hours, min((g + 1) * group_hours, T)
        p = preds[:, lo:hi].reshape(-1)
        o = obs[:, lo:hi].reshape(-1)
        results.append({
            "group": g + 1,
            "hour_range": f"{lo + 1}-{hi}h",
            "nse": round(nse_single(o, p), 4),
            "n": int(p.size),
        })
    return results


def metrics_at_specific_horizons(
    preds: np.ndarray,   # (N, T) point forecast, m3/s
    obs: np.ndarray,     # (N, T)
    horizons: list = None,
) -> dict:
    """NSE, RMSE, MAE, RSE riêng cho từng mốc: 3h, 6h, 12h, 24h, 3d, 7d."""
    if horizons is None:
        horizons = [3, 6, 12, 24, 72, 168]

    horizon_labels = {3: "3h", 6: "6h", 12: "12h", 24: "24h", 72: "3d", 168: "7d"}

    results = {}
    N, T = preds.shape
    for h in horizons:
        label = horizon_labels.get(h, f"{h}h")
        idx = min(h - 1, T - 1)
        if idx >= 0:
            o_h = obs[:, idx]
            p_h = preds[:, idx]
            ss_res = float(np.sum((o_h - p_h) ** 2))
            ss_tot = float(np.sum((o_h - o_h.mean()) ** 2))
            nse_h = 1.0 - ss_res / ss_tot if ss_tot >= 1e-8 else float("nan")
            rmse_h = float(np.sqrt(np.mean((o_h - p_h) ** 2)))
            mae_h = float(np.mean(np.abs(o_h - p_h)))
            rse_h = float(ss_res / max(ss_tot, 1e-8))
            results[label] = {
                "nse": round(nse_h, 4), "rmse": round(rmse_h, 2),
                "mae": round(mae_h, 2), "rse": round(rse_h, 4),
            }
    return results


def picp(obs: np.ndarray, pred_low: np.ndarray, pred_high: np.ndarray) -> dict:
    """PICP (Prediction Interval Coverage Probability) -- ty le % thoi diem
    gia tri thuc te nam trong [pred_low, pred_high] (vd P10-P90). Ly tuong
    PICP ~ 0.80 voi P10/P90 -- khong phai 1.0."""
    obs = np.asarray(obs)
    pred_low = np.asarray(pred_low)
    pred_high = np.asarray(pred_high)
    inside = (obs >= pred_low) & (obs <= pred_high)
    width = pred_high - pred_low
    return {
        "picp": round(float(inside.mean()), 4),
        "mean_interval_width": round(float(width.mean()), 2),
    }


def detect_flood_events(
    obs: np.ndarray,
    threshold: float,
    min_separation: int = 24,
) -> list:
    """Tách các trận lũ riêng lẻ khỏi 1 chuỗi quan trắc liên tục."""
    T = len(obs)
    candidate = np.where(obs >= threshold)[0]
    if len(candidate) == 0:
        return []

    peak_indices = []
    i = 0
    while i < len(candidate):
        j = i
        while j + 1 < len(candidate) and candidate[j + 1] - candidate[j] <= min_separation:
            j += 1
        segment = candidate[i:j + 1]
        peak_indices.append(int(segment[np.argmax(obs[segment])]))
        i = j + 1

    events = []
    for peak_idx in peak_indices:
        start = peak_idx
        while start > 0 and obs[start - 1] <= obs[start]:
            start -= 1
        end = peak_idx
        while end + 1 < T and obs[end + 1] <= obs[end]:
            end += 1
        events.append({"start": start, "peak": peak_idx, "end": end})
    return events


def flood_event_diagnostics(
    obs: np.ndarray,
    pred: np.ndarray,
    threshold: float,
    min_separation: int = 24,
    peak_re_tolerance: float = 0.2,
) -> dict:
    """Chẩn đoán từng trận lũ riêng lẻ (NSE + sai số đỉnh + QA pass rate)."""
    events = detect_flood_events(obs, threshold, min_separation)
    if not events:
        return {
            "n_events": 0, "mean_event_nse": float("nan"),
            "peak_re_mean": float("nan"), "qa_pass_rate": float("nan"),
            "events": [],
        }

    details = []
    for ev in events:
        s, p, e = ev["start"], ev["peak"], ev["end"]
        o_seg = obs[s:e + 1]
        p_seg = pred[s:e + 1]
        ev_nse = nse_single(o_seg, p_seg)

        obs_peak = float(obs[p])
        pred_peak_in_window = float(p_seg.max()) if len(p_seg) else float("nan")
        re = abs(pred_peak_in_window - obs_peak) / max(obs_peak, 1e-6)

        details.append({
            "start": s, "peak": p, "end": e,
            "obs_peak": round(obs_peak, 2),
            "pred_peak": round(pred_peak_in_window, 2),
            "peak_re": round(re, 4),
            "event_nse": round(ev_nse, 4) if not np.isnan(ev_nse) else None,
            "qa_pass": bool(re < peak_re_tolerance),
        })

    valid_nse = [d["event_nse"] for d in details if d["event_nse"] is not None]
    return {
        "n_events": len(details),
        "mean_event_nse": round(float(np.mean(valid_nse)), 4) if valid_nse else float("nan"),
        "peak_re_mean": round(float(np.mean([d["peak_re"] for d in details])), 4),
        "qa_pass_rate": round(float(np.mean([d["qa_pass"] for d in details])), 4),
        "events": details,
    }


## training/train_xgb.py

In [ ]:
# training/train_xgb.py
"""
Huan luyen mo hinh XGBoost theo cac cap do:
  1. Global (16 ho)
  2. Theo Nhanh song (A Vuong, Song Bung, Dak Mi, Song Tranh, Song Con 2 variants)
  3. Theo Luu vuc song (Vu Gia, Thu Bon)
  4. Theo Tung ho rieng le
  5. Theo Mua (Mua kho: T1-8, Mua mua: T9-12, Ca nam: all)

Chay:
    python training/train_xgb.py --mode branch --branch A_VUONG
    python training/train_xgb.py --mode branch --branch SONG_BUNG --season rainy
    python training/train_xgb.py --mode basin --basin "Vu Gia"
    python training/train_xgb.py --mode all-branches
    python training/train_xgb.py --mode all-basins
    python training/train_xgb.py --mode single --rid 2
"""
import os
import sys
import time
import argparse
import numpy as np
import xgboost as xgb

if sys.stdout.encoding is not None and sys.stdout.encoding.lower() != "utf-8":
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")
    sys.stderr.reconfigure(encoding="utf-8", errors="replace")




ARTIFACT_DIR = "artifacts/xgb"


def flood_sample_weight(y_train: np.ndarray, ts_train: np.ndarray, season: str = "all") -> np.ndarray:
    """Trong so mau theo bien do dinh lu va theo mua."""
    peak = y_train.max(axis=1)
    w = np.ones(len(y_train), dtype=np.float32)
    thr_95 = np.percentile(peak, 95)
    thr_99 = np.percentile(peak, 99)
    w[peak >= thr_95] = 2.0
    w[peak >= thr_99] = 3.0

    if season == "all":
        months = ts_train.astype("datetime64[M]").astype(int) % 12 + 1
        is_rainy = np.isin(months, [9, 10, 11, 12])
        w[is_rainy] *= RAINY_SEASON_WEIGHT
    return w


def train_xgb_dataset(X: np.ndarray, y: np.ndarray, ts: np.ndarray,
                      artifact_dir: str, season: str = "all",
                      init_boosters: dict = None,
                      num_boost_round: int = 2000,
                      early_stopping_rounds: int = 50):
    """Huấn luyện 72 booster cho 1 tập dữ liệu cụ thể.

    init_boosters: dict {(h, q): xgb.Booster} đã train sẵn (vd model nhánh/lưu
    vực) -- nếu truyền vào, mỗi booster (h,q) sẽ CONTINUE boosting (xgb.train
    xgb_model=...) trên dữ liệu X/y/ts của hàm này thay vì train từ đầu. Đây là
    kịch bản 5 (transfer learning) áp dụng cho XGBoost: pretrain trên nhánh/lưu
    vực (init_boosters=None, num_boost_round lớn) rồi fine-tune riêng từng hồ
    (init_boosters=<72 booster nhánh>, num_boost_round nhỏ hơn -- ví dụ 300).
    """
    train_idx, val_idx, test_idx = split_60_20_20(ts)
    months = ts.astype("datetime64[M]").astype(int) % 12 + 1

    if season == "dry":
        season_mask = np.isin(months, [1, 2, 3, 4, 5, 6, 7, 8])
        train_idx = [i for i in train_idx if season_mask[i]]
        val_idx   = [i for i in val_idx if season_mask[i]]
    elif season == "rainy":
        season_mask = np.isin(months, [9, 10, 11, 12])
        train_idx = [i for i in train_idx if season_mask[i]]
        val_idx   = [i for i in val_idx if season_mask[i]]

    print(f"Dataset: Total={len(X):,} | Train={len(train_idx):,} | Val={len(val_idx):,} | Season={season.upper()}")
    if len(train_idx) == 0 or len(val_idx) == 0:
        print("  WARNING: Train/Val rong, bo qua.")
        return

    sample_weight = flood_sample_weight(y[train_idx], ts[train_idx], season=season)
    os.makedirs(artifact_dir, exist_ok=True)

    params_base = dict(
        tree_method="hist",
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=5,
        reg_lambda=1.0,
    )

    t0 = time.time()
    n_trained = 0
    for h in range(HORIZON):
        dtrain = xgb.DMatrix(X[train_idx], label=y[train_idx, h], weight=sample_weight)
        dval = xgb.DMatrix(X[val_idx], label=y[val_idx, h])

        for q in QUANTILES:
            params = {**params_base, "objective": "reg:quantileerror", "quantile_alpha": q}
            xgb_model = init_boosters.get((h, q)) if init_boosters else None
            bst = xgb.train(
                params, dtrain,
                num_boost_round=num_boost_round,
                evals=[(dval, "val")],
                early_stopping_rounds=early_stopping_rounds,
                verbose_eval=False,
                xgb_model=xgb_model,
            )
            bst.save_model(f"{artifact_dir}/h{h + 1:02d}_q{int(q * 100):02d}.json")
            n_trained += 1

    elapsed = time.time() - t0
    print(f"  [OK] Da train {n_trained} boosters trong {elapsed:.1f}s -> {artifact_dir}/\n")


def filter_by_rids(X: np.ndarray, y: np.ndarray, rid_arr: np.ndarray, ts_arr: np.ndarray, target_rids: list):
    """Lọc dữ liệu theo danh sách reservoir IDs."""
    idx_map = {info["idx"]: rid for rid, info in RESERVOIRS.items()}
    actual_rids = np.array([idx_map.get(r, -1) for r in rid_arr])
    mask = np.isin(actual_rids, target_rids)
    return X[mask], y[mask], rid_arr[mask], ts_arr[mask]


def main():
    parser = argparse.ArgumentParser(description="Huấn luyện XGBoost theo Cấp độ và Mùa")
    parser.add_argument("--mode", type=str, default="global",
                        choices=["global", "branch", "basin", "single", "all-branches", "all-basins", "all-single"],
                        help="Che do train")
    parser.add_argument("--branch", type=str, default=None,
                        help="Tên nhánh: A_VUONG, SONG_BUNG, DAK_MI, SONG_TRANH, A_VUONG_WITH_SONG_CON, SONG_BUNG_WITH_SONG_CON")
    parser.add_argument("--basin", type=str, default=None,
                        help="Tên lưu vực: 'Vu Gia' hoặc 'Thu Bồn'")
    parser.add_argument("--rid", type=int, default=None, help="Reservoir ID (vd: 1, 2, 3...)")
    parser.add_argument("--season", type=str, default="all", choices=["all", "dry", "rainy"],
                        help="Mùa: all, dry (T1-8), rainy (T9-12)")
    args = parser.parse_args()

    print("=" * 70)
    print(f"HUẤN LUYỆN XGBOOST | MODE: {args.mode.upper()} | MÙA: {args.season.upper()}")
    print("=" * 70)

    X_all, y_all, rid_all, ts_all = build_tabular_dataset()
    season_suffix = f"_{args.season}" if args.season != "all" else ""

    if args.mode == "global":
        art_dir = f"artifacts/xgb{season_suffix}"
        train_xgb_dataset(X_all, y_all, ts_all, art_dir, season=args.season)

    elif args.mode == "branch" and args.branch:
        b_key = args.branch.upper()
        if b_key in RIVER_BRANCHES:
            rids = RIVER_BRANCHES[b_key]
        elif b_key == "A_VUONG_WITH_SONG_CON":
            rids = SONG_CON_2_VARIANTS["A_VUONG"]
        elif b_key == "SONG_BUNG_WITH_SONG_CON":
            rids = SONG_CON_2_VARIANTS["SONG_BUNG"]
        else:
            raise ValueError(f"branch={args.branch} không hợp lệ.")
        X_b, y_b, _, ts_b = filter_by_rids(X_all, y_all, rid_all, ts_all, rids)
        art_dir = f"artifacts/xgb_branch/{b_key}{season_suffix}"
        train_xgb_dataset(X_b, y_b, ts_b, art_dir, season=args.season)

    elif args.mode == "all-branches":
        branch_list = list(RIVER_BRANCHES.keys()) + ["A_VUONG_WITH_SONG_CON", "SONG_BUNG_WITH_SONG_CON"]
        for b_name in branch_list:
            if b_name in RIVER_BRANCHES:
                rids = RIVER_BRANCHES[b_name]
            elif b_name == "A_VUONG_WITH_SONG_CON":
                rids = SONG_CON_2_VARIANTS["A_VUONG"]
            else:
                rids = SONG_CON_2_VARIANTS["SONG_BUNG"]
            print(f"\n>>> TRAIN NHÁNH: {b_name} ({len(rids)} hồ)")
            X_b, y_b, _, ts_b = filter_by_rids(X_all, y_all, rid_all, ts_all, rids)
            art_dir = f"artifacts/xgb_branch/{b_name}{season_suffix}"
            train_xgb_dataset(X_b, y_b, ts_b, art_dir, season=args.season)

    elif args.mode == "basin" and args.basin:
        rids = RIVER_BASINS_EXPERIMENT.get(args.basin)
        if not rids:
            raise ValueError(f"basin={args.basin} không hợp lệ.")
        b_clean = args.basin.upper().replace(" ", "_")
        X_b, y_b, _, ts_b = filter_by_rids(X_all, y_all, rid_all, ts_all, rids)
        art_dir = f"artifacts/xgb_basin/{b_clean}{season_suffix}"
        train_xgb_dataset(X_b, y_b, ts_b, art_dir, season=args.season)

    elif args.mode == "all-basins":
        for basin_name, rids in RIVER_BASINS_EXPERIMENT.items():
            b_clean = basin_name.upper().replace(" ", "_")
            print(f"\n>>> TRAIN LƯU VỰC: {basin_name} ({len(rids)} hồ)")
            X_b, y_b, _, ts_b = filter_by_rids(X_all, y_all, rid_all, ts_all, rids)
            art_dir = f"artifacts/xgb_basin/{b_clean}{season_suffix}"
            train_xgb_dataset(X_b, y_b, ts_b, art_dir, season=args.season)

    elif args.mode == "single" and args.rid is not None:
        info = RESERVOIRS[args.rid]
        res_key = info["name"].replace(" ", "_")
        X_s, y_s, _, ts_s = filter_by_rids(X_all, y_all, rid_all, ts_all, [args.rid])
        art_dir = f"artifacts/xgb_single/{res_key}{season_suffix}"
        train_xgb_dataset(X_s, y_s, ts_s, art_dir, season=args.season)

    elif args.mode == "all-single":
        for rid, info in RESERVOIRS.items():
            res_key = info["name"].replace(" ", "_")
            print(f"\n>>> TRAIN TỪNG HỒ: {info['name']} (RID={rid})")
            X_s, y_s, _, ts_s = filter_by_rids(X_all, y_all, rid_all, ts_all, [rid])
            art_dir = f"artifacts/xgb_single/{res_key}{season_suffix}"
            train_xgb_dataset(X_s, y_s, ts_s, art_dir, season=args.season)


## training/evaluate_xgb.py

In [ ]:
# training/evaluate_xgb.py
"""
Danh gia mo hinh XGBoost tren tap test 2025 (flood season holdout)
Dung event_metrics.py (NSE, KGE, R2, RMSE/MAE, PICP cho quantile coverage)
Kem phan ra danh gia theo Mua Kho (T1-8) va Mua Mua (T9-12).

Chay:
    python training/evaluate_xgb.py                                      # Global model
    python training/evaluate_xgb.py --artifact-dir artifacts/xgb_branch/A_VUONG
    python training/evaluate_xgb.py --artifact-dir artifacts/xgb_basin/VU_GIA
"""
import os
import sys
import json
import argparse
import numpy as np
import pandas as pd
import xgboost as xgb

if sys.stdout.encoding is not None and sys.stdout.encoding.lower() != "utf-8":
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")
    sys.stderr.reconfigure(encoding="utf-8", errors="replace")






def load_boosters(artifact_dir: str):
    models = {}
    for h in range(HORIZON):
        for q in QUANTILES:
            path = os.path.join(artifact_dir, f"h{h + 1:02d}_q{int(q * 100):02d}.json")
            if not os.path.exists(path):
                raise FileNotFoundError(f"Không tìm thấy file model: {path}")
            bst = xgb.Booster()
            bst.load_model(path)
            models[(h, q)] = bst
    return models


def predict_all(models: dict, X: np.ndarray) -> np.ndarray:
    """Tra ve (N, HORIZON, n_quantiles), da sort de chong quantile crossing."""
    N = X.shape[0]
    preds = np.zeros((N, HORIZON, len(QUANTILES)), dtype=np.float32)
    dmat = xgb.DMatrix(X)
    for h in range(HORIZON):
        for qi, q in enumerate(QUANTILES):
            preds[:, h, qi] = models[(h, q)].predict(dmat)
    preds = np.sort(preds, axis=2)
    return preds


def evaluate(artifact_dir: str = "artifacts/xgb", output_prefix: str = "xgb", json_out_dir: str = None,
             data: tuple = None):
    """json_out_dir: nếu truyền vào, lưu thêm 1 file <Ten_Ho>.json / hồ (đầy đủ
    metric: nse/nse_dry/nse_rainy/kge/r2/mae/rmse/horizons/picp) -- dùng để
    main_generate_excel_summary.py gộp nhiều lần evaluate() (single/branch/
    basin x season) lại thành 1 bảng so sánh, không phải đọc lại từ Excel.

    data: (X, y, rid, ts) đã build sẵn (build_tabular_dataset()) -- truyền vào
    khi gọi evaluate() NHIỀU LẦN liên tiếp (vd notebook master train+eval tuần
    tự single/nhánh/lưu vực/fine-tune x mùa, ~70 lần gọi) để khỏi build lại
    dataset từ đĩa mỗi lần (rất tốn thời gian). None = tự build (dùng CLI)."""
    print("=" * 70)
    print(f"ĐÁNH GIÁ XGBOOST MODEL: {artifact_dir}")
    print("=" * 70)

    X, y, rid, ts = data if data is not None else build_tabular_dataset()
    _, _, test_idx = split_60_20_20(ts)
    if len(test_idx) == 0:
        raise RuntimeError("Test set rong -- kiem tra du lieu >= TEST_START.")

    print(f"Test samples: {len(test_idx):,}")
    models = load_boosters(artifact_dir)

    preds = predict_all(models, X[test_idx])
    med_idx = len(QUANTILES) // 2
    preds_med = preds[:, :, med_idx]
    preds_p10 = preds[:, :, 0]
    preds_p90 = preds[:, :, -1]

    preds_raw   = np.clip(preds_med, 0, None) ** 2
    p10_raw     = np.clip(preds_p10, 0, None) ** 2
    p90_raw     = np.clip(preds_p90, 0, None) ** 2
    targets_raw = y[test_idx] ** 2
    rids_test   = rid[test_idx]
    ts_test     = ts[test_idx]

    test_months = ts_test.astype("datetime64[M]").astype(int) % 12 + 1
    dry_mask_all = np.isin(test_months, [1, 2, 3, 4, 5, 6, 7, 8])
    rainy_mask_all = np.isin(test_months, [9, 10, 11, 12])

    idx_to_name = {info["idx"]: info["name"] for _, info in RESERVOIRS.items()}

    results_rows = []
    nse_dict = {}
    for r_idx in sorted(idx_to_name.keys()):
        mask = rids_test == r_idx
        if not mask.any():
            continue
        p_r, t_r = preds_raw[mask].reshape(-1), targets_raw[mask].reshape(-1)
        nse = nse_single(t_r, p_r)
        if np.isnan(nse):
            continue
        nse_dict[r_idx] = nse
        kge = kge_single(t_r, p_r)
        r2 = r2_single(t_r, p_r)
        mae = float(np.mean(np.abs(p_r - t_r)))
        rmse = float(np.sqrt(np.mean((p_r - t_r) ** 2)))
        cov = picp(targets_raw[mask].reshape(-1), p10_raw[mask].reshape(-1), p90_raw[mask].reshape(-1))

        # Phân rã theo mùa
        mask_dry = mask & dry_mask_all
        mask_rainy = mask & rainy_mask_all
        nse_dry = nse_single(targets_raw[mask_dry].reshape(-1), preds_raw[mask_dry].reshape(-1)) if mask_dry.any() else np.nan
        nse_rainy = nse_single(targets_raw[mask_rainy].reshape(-1), preds_raw[mask_rainy].reshape(-1)) if mask_rainy.any() else np.nan
        rmse_dry = float(np.sqrt(np.mean((preds_raw[mask_dry] - targets_raw[mask_dry]) ** 2))) if mask_dry.any() else np.nan
        rmse_rainy = float(np.sqrt(np.mean((preds_raw[mask_rainy] - targets_raw[mask_rainy]) ** 2))) if mask_rainy.any() else np.nan

        # Phân rã theo mốc lead-time cụ thể (3h/6h/12h/24h)
        horizons = {}
        for hz in (3, 6, 12, 24):
            if hz > preds_raw.shape[1]:
                continue
            p_h, t_h = preds_raw[mask, hz - 1], targets_raw[mask, hz - 1]
            nse_h = nse_single(t_h, p_h)
            rmse_h = float(np.sqrt(np.mean((p_h - t_h) ** 2)))
            horizons[f"{hz}h"] = {"nse": round(nse_h, 4) if not np.isnan(nse_h) else None, "rmse": round(rmse_h, 2)}

        name = idx_to_name[r_idx]
        row_metrics = {
            "reservoir": name, "nse": round(nse, 4), "kge": round(kge, 4), "r2": round(r2, 4),
            "mae": round(mae, 2), "rmse": round(rmse, 2),
            "nse_dry_season": round(nse_dry, 4) if not np.isnan(nse_dry) else None,
            "nse_rainy_season": round(nse_rainy, 4) if not np.isnan(nse_rainy) else None,
            "rmse_dry_season": round(rmse_dry, 2) if not np.isnan(rmse_dry) else None,
            "rmse_rainy_season": round(rmse_rainy, 2) if not np.isnan(rmse_rainy) else None,
            "picp_p10_p90": cov["picp"], "mean_interval_width": cov["mean_interval_width"],
            "horizons": horizons,
        }
        results_rows.append({
            "Reservoir": name,
            "NSE (Cả năm)": round(nse, 4),
            "NSE (Mùa khô T1-8)": round(nse_dry, 4) if not np.isnan(nse_dry) else "N/A",
            "NSE (Mùa mưa T9-12)": round(nse_rainy, 4) if not np.isnan(nse_rainy) else "N/A",
            "KGE": round(kge, 4),
            "R2": round(r2, 4),
            "MAE (m³/s)": round(mae, 2),
            "RMSE (m³/s)": round(rmse, 2),
            "PICP (P10-P90)": cov["picp"],
            "Mean Interval Width": cov["mean_interval_width"],
        })
        print(f"  {name:.<28} NSE={nse:.3f} | NSE_Dry={nse_dry:.3f} | NSE_Rainy={nse_rainy:.3f} | KGE={kge:.3f}")

        if json_out_dir:
            os.makedirs(json_out_dir, exist_ok=True)
            with open(os.path.join(json_out_dir, f"{name.replace(' ', '_')}.json"), "w", encoding="utf-8") as f:
                json.dump(row_metrics, f, ensure_ascii=False, indent=2)

    avg_nse = sum(nse_dict.values()) / len(nse_dict) if nse_dict else 0.0
    results_rows.append({"Reservoir": "--- AVERAGE ---", "NSE (Cả năm)": round(avg_nse, 4)})
    print(f"\n  NSE AVERAGE ({len(nse_dict)} hồ): {avg_nse:.3f}")

    excel_out = f"ket_qua_danh_gia_2025_{output_prefix}.xlsx"
    pd.DataFrame(results_rows).to_excel(excel_out, index=False)
    print(f"\nSaved: {excel_out}")


## Build dataset 1 lần duy nhất (tái dùng cho toàn bộ train + evaluate)

In [ ]:
X_all, y_all, rid_all, ts_all = build_tabular_dataset()
DATA_ALL = (X_all, y_all, rid_all, ts_all)


## Helper: hồ -> nhánh sông / lưu vực nó thuộc về, tiện ích đặt tên theo mùa

In [ ]:
BRANCH_GROUPS = list(RIVER_BRANCHES.keys()) + ["A_VUONG_WITH_SONG_CON", "SONG_BUNG_WITH_SONG_CON"]
BASIN_KEY = {"Vu Gia": "VU_GIA", "Thu Bồn": "THU_BON"}

def own_branch(rid):
    for b, rl in RIVER_BRANCHES.items():
        if rid in rl:
            return b
    return None

def own_basin(rid):
    for b, rl in RIVER_BASINS_EXPERIMENT.items():
        if rid in rl:
            return b
    return None

def rids_for_branch(b_name):
    if b_name in RIVER_BRANCHES:
        return RIVER_BRANCHES[b_name]
    if b_name == "A_VUONG_WITH_SONG_CON":
        return SONG_CON_2_VARIANTS["A_VUONG"]
    if b_name == "SONG_BUNG_WITH_SONG_CON":
        return SONG_CON_2_VARIANTS["SONG_BUNG"]
    raise ValueError(b_name)

def season_group(base, season):
    return base if season == "all" else f"{base}_{season}"

def is_done(art_dir):
    return os.path.exists(f"{art_dir}/h24_q90.json")


## Pha 1 — Single (train riêng từng hồ, baseline gốc, KHÔNG warm-start) x 3 mùa

In [ ]:
for rid, info in RESERVOIRS.items():
    key = info["name"].replace(" ", "_")
    for season in SEASONS:
        art_dir = f"artifacts/xgb_single/{season_group(key, season)}"
        if SKIP_IF_DONE and is_done(art_dir):
            print(f"[SKIP/RESUME] Single {info['name']} ({season}) đã train.")
            continue
        print(f"\n>>> PHA 1 - TRAIN SINGLE: {info['name']} | MÙA {season.upper()}")
        X_s, y_s, _, ts_s = filter_by_rids(X_all, y_all, rid_all, ts_all, [rid])
        train_xgb_dataset(X_s, y_s, ts_s, art_dir, season=season)


In [ ]:
for rid, info in RESERVOIRS.items():
    key = info["name"].replace(" ", "_")
    for season in SEASONS:
        group = season_group(key, season)
        art_dir = f"artifacts/xgb_single/{group}"
        json_dir = f"eval_json/single/{group}"
        if not is_done(art_dir):
            continue
        if SKIP_IF_DONE and os.path.exists(f"{json_dir}/{key}.json"):
            continue
        evaluate(artifact_dir=art_dir, output_prefix=f"single_{group}", json_out_dir=json_dir, data=DATA_ALL)


## Pha 2 — Nhánh sông (4 nhánh + 2 biến thể Sông Côn 2) x 3 mùa — Kịch bản 1 + 2 + 4

In [ ]:
for b_name in BRANCH_GROUPS:
    for season in SEASONS:
        group = season_group(b_name, season)
        art_dir = f"artifacts/xgb_branch/{group}"
        if SKIP_IF_DONE and is_done(art_dir):
            print(f"[SKIP/RESUME] Nhánh {b_name} ({season}) đã train.")
            continue
        print(f"\n>>> PHA 2 - TRAIN NHÁNH: {b_name} | MÙA {season.upper()}")
        X_b, y_b, _, ts_b = filter_by_rids(X_all, y_all, rid_all, ts_all, rids_for_branch(b_name))
        train_xgb_dataset(X_b, y_b, ts_b, art_dir, season=season)


In [ ]:
for b_name in BRANCH_GROUPS:
    for season in SEASONS:
        group = season_group(b_name, season)
        art_dir = f"artifacts/xgb_branch/{group}"
        json_dir = f"eval_json/branch/{group}"
        if not is_done(art_dir):
            continue
        if SKIP_IF_DONE and os.path.isdir(json_dir) and len(os.listdir(json_dir)) > 0:
            continue
        evaluate(artifact_dir=art_dir, output_prefix=f"branch_{group}", json_out_dir=json_dir, data=DATA_ALL)


## Pha 3 — Lưu vực sông (2 lưu vực thực nghiệm) x 3 mùa — Kịch bản 3 + 4
Vu Gia = nhánh Sông Tranh (2/3/4) + Khe Diên. Thu Bồn = 12 hồ còn lại
(đúng định nghĩa người dùng yêu cầu).


In [ ]:
for basin_name, rids in RIVER_BASINS_EXPERIMENT.items():
    bkey = BASIN_KEY[basin_name]
    for season in SEASONS:
        group = season_group(bkey, season)
        art_dir = f"artifacts/xgb_basin/{group}"
        if SKIP_IF_DONE and is_done(art_dir):
            print(f"[SKIP/RESUME] Lưu vực {basin_name} ({season}) đã train.")
            continue
        print(f"\n>>> PHA 3 - TRAIN LƯU VỰC: {basin_name} | MÙA {season.upper()}")
        X_b, y_b, _, ts_b = filter_by_rids(X_all, y_all, rid_all, ts_all, rids)
        train_xgb_dataset(X_b, y_b, ts_b, art_dir, season=season)


In [ ]:
for basin_name in RIVER_BASINS_EXPERIMENT:
    bkey = BASIN_KEY[basin_name]
    for season in SEASONS:
        group = season_group(bkey, season)
        art_dir = f"artifacts/xgb_basin/{group}"
        json_dir = f"eval_json/basin/{group}"
        if not is_done(art_dir):
            continue
        if SKIP_IF_DONE and os.path.isdir(json_dir) and len(os.listdir(json_dir)) > 0:
            continue
        evaluate(artifact_dir=art_dir, output_prefix=f"basin_{group}", json_out_dir=json_dir, data=DATA_ALL)


## Pha 4 — Fine-tune riêng từng hồ bằng CONTINUE-BOOSTING, x 3 mùa (Kịch bản 5)
Tương đương Transfer Learning bên LSTM: nạp 72 booster ĐÚNG MÙA đã train ở
Pha 2/3 (`xgb_model=...`), boost thêm trên dữ liệu 1 hồ ĐÃ LỌC CÙNG MÙA
thay vì train lại từ đầu. Chạy sau Pha 2+3 vì cần booster dry/rainy sẵn có.


In [ ]:
for rid, info in RESERVOIRS.items():
    key = info["name"].replace(" ", "_")
    b_own = own_branch(rid)
    basin_own = own_basin(rid)

    for season in SEASONS:
        X_s, y_s, _, ts_s = filter_by_rids(X_all, y_all, rid_all, ts_all, [rid])

        if b_own:
            branch_art_dir = f"artifacts/xgb_branch/{season_group(b_own, season)}"
            ft_art_dir = f"artifacts/xgb_finetune_branch/{season_group(key, season)}"
            if is_done(branch_art_dir) and not (SKIP_IF_DONE and is_done(ft_art_dir)):
                print(f"\n>>> PHA 4 - FINE-TUNE {info['name']} từ nhánh {b_own} | MÙA {season.upper()}")
                init_boosters = load_boosters(branch_art_dir)
                train_xgb_dataset(X_s, y_s, ts_s, ft_art_dir, season=season,
                                   init_boosters=init_boosters, num_boost_round=300, early_stopping_rounds=30)

        if basin_own:
            bkey = BASIN_KEY[basin_own]
            basin_art_dir = f"artifacts/xgb_basin/{season_group(bkey, season)}"
            ft_art_dir = f"artifacts/xgb_finetune_basin/{season_group(key, season)}"
            if is_done(basin_art_dir) and not (SKIP_IF_DONE and is_done(ft_art_dir)):
                print(f"\n>>> PHA 4 - FINE-TUNE {info['name']} từ lưu vực {basin_own} | MÙA {season.upper()}")
                init_boosters = load_boosters(basin_art_dir)
                train_xgb_dataset(X_s, y_s, ts_s, ft_art_dir, season=season,
                                   init_boosters=init_boosters, num_boost_round=300, early_stopping_rounds=30)


In [ ]:
for rid, info in RESERVOIRS.items():
    key = info["name"].replace(" ", "_")
    for method in ["finetune_branch", "finetune_basin"]:
        for season in SEASONS:
            group = season_group(key, season)
            art_dir = f"artifacts/xgb_{method}/{group}"
            json_dir = f"eval_json/{method}/{group}"
            if not is_done(art_dir):
                continue
            if SKIP_IF_DONE and os.path.exists(f"{json_dir}/{key}.json"):
                continue
            evaluate(artifact_dir=art_dir, output_prefix=f"{method}_{group}", json_out_dir=json_dir, data=DATA_ALL)


## Pha 5 — Tổng hợp toàn bộ kết quả ra 1 file Excel

In [ ]:
# main_generate_excel_summary.py
"""
Tổng hợp kết quả XGBoost theo ĐÚNG 5 phương pháp x 3 mùa (song song với bản
LSTM ở LSTM_Py_Backend_v2/main_generate_excel_summary.py, cùng layout khái
niệm):
  A. Single       -- train riêng từng hồ (72 booster/hồ), không warm-start
  B1. Nhánh sông   -- 72 booster pooled 4 nhánh (+ 2 biến thể Sông Côn 2)
  B2. Lưu vực sông -- 72 booster pooled 2 lưu vực thực nghiệm
  C1. Fine-tune Nhánh -- continue-boosting (xgb_model=...) từ B1, tiếp tục train
      thêm trên riêng dữ liệu 1 hồ (tương đương Transfer Learning bên LSTM)
  C2. Fine-tune Lưu vực -- continue-boosting từ B2
  Mỗi phương pháp đều có 3 biến thể mùa: Cả năm / Mùa Khô (T1-8) / Mùa Mưa
  (T9-12) -- xem kaggle/generate_notebook_master.py cho toàn bộ pipeline.

Đọc từ eval_json/<method>/<group>[_dry|_rainy]/<Ten_Ho>.json do
training/evaluate_xgb.py::evaluate(..., json_out_dir=...) ghi ra.

Chạy: python main_generate_excel_summary.py
"""
import os
import sys
import json
import numpy as np
import pandas as pd

try:
    sys.stdout.reconfigure(encoding="utf-8")
except Exception:
    pass

try:
    ROOT_DEFAULT = os.path.dirname(os.path.abspath(__file__))
except NameError:
    ROOT_DEFAULT = os.getcwd()



CUSTOM_ORDER = [
    "HO ZA HUNG", "HO DAK MI 3", "HO SONG BUNG 4", "HO DAK MI 2", "HO DAK MI 4",
    "HO SONG TRANH 4", "HO A VUONG", "HO SONG TRANH 3", "HO SONG TRANH 2",
    "HO SONG BUNG 2", "HO SONG CON 2", "HO KHE DIEN", "HO SONG BUNG 5",
    "HO SONG BUNG 6", "HO SONG BUNG 4A", "HO DAK MI 4C",
]

BASIN_KEY = {"Vu Gia": "VU_GIA", "Thu Bồn": "THU_BON"}
SEASON_LABEL = {"all": "Cả năm", "dry": "Mùa Khô (model chuyên mùa)", "rainy": "Mùa Mưa (model chuyên mùa)"}


def get_branch_for_reservoir(rid: int) -> str:
    for b_name, r_list in RIVER_BRANCHES.items():
        if rid in r_list:
            return b_name
    if rid == 16:
        return "SONG_CON_2 (A_VUONG/SONG_BUNG)"
    return "KHAC"


def get_basin_for_reservoir(rid: int) -> str:
    for b_name, r_list in RIVER_BASINS_EXPERIMENT.items():
        if rid in r_list:
            return b_name
    return "KHAC"


def _load_json(path: str) -> dict:
    if path and os.path.exists(path):
        try:
            with open(path, "r", encoding="utf-8") as f:
                return json.load(f)
        except Exception:
            return {}
    return {}


def _season_group(base: str, season: str) -> str:
    return base if season == "all" else f"{base}_{season}"


def _eval_path(root: str, method: str, group: str, key: str) -> str:
    """<root>/eval_json/<method>/<group>/<key>.json"""
    return os.path.join(root, "eval_json", method, group, f"{key}.json")


def _fnum(m: dict, field: str = "nse"):
    v = m.get(field, float("nan")) if m else float("nan")
    return float("nan") if v is None else v


def _season_field(m: dict, season: str, base: str = "nse"):
    """season='all' -> field <base> (tổng, model train cả năm).
    season='dry'/'rainy' -> field <base>_dry_season/<base>_rainy_season (model
    CHUYÊN train riêng mùa đó, đo đúng trên lát mùa nó được train)."""
    if season == "all":
        return _fnum(m, base)
    return _fnum(m, f"{base}_{season}_season")


def _fmt(v):
    return round(v, 4) if isinstance(v, (int, float)) and not np.isnan(v) else "N/A"


def generate_summary(root: str = None) -> str:
    root = root or ROOT_DEFAULT
    print("=" * 90)
    print("XGBOOST — TỔNG HỢP SO SÁNH 5 PHƯƠNG PHÁP x 3 MÙA (CẢ NĂM / KHÔ / MƯA)")
    print(f"Đọc dữ liệu từ: {root}")
    print("=" * 90)

    name_to_rid = {info["name"]: rid for rid, info in RESERVOIRS.items()}
    main_rows, detail_rows, horizon_rows = [], [], []

    for res_name in CUSTOM_ORDER:
        rid = name_to_rid.get(res_name)
        if not rid:
            continue
        info = RESERVOIRS[rid]
        key = info["name"].replace(" ", "_")
        branch_name = get_branch_for_reservoir(rid)
        basin_name = get_basin_for_reservoir(rid)
        bkey = BASIN_KEY.get(basin_name, basin_name)

        # ── Bảng đầu: chỉ số Cả năm của 5 phương pháp (headline so sánh) ──────
        m_single = _load_json(_eval_path(root, "single", key, key))
        m_branch = _load_json(_eval_path(root, "branch", branch_name, key))
        m_basin = _load_json(_eval_path(root, "basin", bkey, key))
        m_ft_branch = _load_json(_eval_path(root, "finetune_branch", key, key))
        m_ft_basin = _load_json(_eval_path(root, "finetune_basin", key, key))

        method_nse = {
            "Single": _fnum(m_single),
            "Nhánh (pooled)": _fnum(m_branch),
            "Lưu vực (pooled)": _fnum(m_basin),
            "Fine-tune Nhánh": _fnum(m_ft_branch),
            "Fine-tune Lưu vực": _fnum(m_ft_basin),
        }
        available = {k: v for k, v in method_nse.items() if not np.isnan(v)}
        if available:
            best_method = max(available, key=available.get)
            verdict = f"Tốt nhất: {best_method} (NSE={available[best_method]:.3f})"
        else:
            best_method, verdict = "N/A", "Chưa có đủ kết quả (chưa train/eval xong)"

        main_rows.append({
            "Hồ Chứa": info["name"],
            "Nhánh Sông": branch_name,
            "Lưu Vực (thực nghiệm)": basin_name,
            "NSE Single": _fmt(method_nse["Single"]),
            "NSE Nhánh (pooled, chưa FT)": _fmt(method_nse["Nhánh (pooled)"]),
            "NSE Lưu vực (pooled, chưa FT)": _fmt(method_nse["Lưu vực (pooled)"]),
            "NSE Fine-tune từ Nhánh": _fmt(method_nse["Fine-tune Nhánh"]),
            "NSE Fine-tune từ Lưu vực": _fmt(method_nse["Fine-tune Lưu vực"]),
            "RMSE Single": _fmt(_fnum(m_single, "rmse")),
            "RMSE Fine-tune Nhánh": _fmt(_fnum(m_ft_branch, "rmse")),
            "RMSE Fine-tune Lưu vực": _fmt(_fnum(m_ft_basin, "rmse")),
            "KGE Fine-tune Nhánh": _fmt(_fnum(m_ft_branch, "kge")),
            "KGE Fine-tune Lưu vực": _fmt(_fnum(m_ft_basin, "kge")),
            "Phương Pháp Tốt Nhất": best_method,
            "Nhận Định": verdict,
        })

        # ── Bảng chi tiết: 5 phương pháp x 3 mùa (dạng dài, dễ pivot trong Excel) ──
        method_loaders = {
            "Single": lambda s: _load_json(_eval_path(root, "single", _season_group(key, s), key)),
            "Nhánh (pooled)": lambda s: _load_json(_eval_path(root, "branch", _season_group(branch_name, s), key)),
            "Lưu vực (pooled)": lambda s: _load_json(_eval_path(root, "basin", _season_group(bkey, s), key)),
            "Fine-tune Nhánh": lambda s: _load_json(_eval_path(root, "finetune_branch", _season_group(key, s), key)),
            "Fine-tune Lưu vực": lambda s: _load_json(_eval_path(root, "finetune_basin", _season_group(key, s), key)),
        }
        for method_name, loader in method_loaders.items():
            for season in ("all", "dry", "rainy"):
                m = loader(season)
                nse_val = _season_field(m, season, "nse")
                rmse_val = _season_field(m, season, "rmse")
                detail_rows.append({
                    "Hồ Chứa": info["name"],
                    "Nhánh Sông": branch_name,
                    "Phương Pháp": method_name,
                    "Mùa": SEASON_LABEL[season],
                    "NSE": _fmt(nse_val),
                    "RMSE (m³/s)": _fmt(rmse_val),
                })

        h_single = m_single.get("horizons", {}) if m_single else {}
        h_ft_branch = m_ft_branch.get("horizons", {}) if m_ft_branch else {}
        h_ft_basin = m_ft_basin.get("horizons", {}) if m_ft_basin else {}
        h_row = {"Hồ Chứa": info["name"]}
        for hz in ["3h", "6h", "12h", "24h"]:
            h_row[f"NSE_Single_{hz}"] = h_single.get(hz, {}).get("nse", "N/A")
            h_row[f"NSE_FT_Nhanh_{hz}"] = h_ft_branch.get(hz, {}).get("nse", "N/A")
            h_row[f"NSE_FT_LuuVuc_{hz}"] = h_ft_basin.get(hz, {}).get("nse", "N/A")
        horizon_rows.append(h_row)

    # ── Sông Côn 2: A Vương vs Sông Bung vs riêng (Kịch bản 2) ──────────────────
    key_sc2 = "HO_SONG_CON_2"
    m_sc2_single = _load_json(_eval_path(root, "single", key_sc2, key_sc2))
    m_sc2_a_vuong_variant = _load_json(_eval_path(root, "branch", "A_VUONG_WITH_SONG_CON", key_sc2))
    m_sc2_song_bung_variant = _load_json(_eval_path(root, "branch", "SONG_BUNG_WITH_SONG_CON", key_sc2))

    song_con_rows = [
        {"Biến Thể": "Single (baseline)", "Các Hồ Cùng Train": "Chỉ Sông Côn 2",
         "NSE": _fmt(_fnum(m_sc2_single))},
        {"Biến Thể": "Thuộc Nhánh A Vương", "Các Hồ Cùng Train": "A Vương, Za Hưng, Sông Côn 2",
         "NSE": _fmt(_fnum(m_sc2_a_vuong_variant))},
        {"Biến Thể": "Thuộc Nhánh Sông Bung", "Các Hồ Cùng Train": "Sông Bung 2/4/4A/5/6, Sông Côn 2",
         "NSE": _fmt(_fnum(m_sc2_song_bung_variant))},
    ]

    df_main = pd.DataFrame(main_rows)
    df_detail = pd.DataFrame(detail_rows)
    df_horizon = pd.DataFrame(horizon_rows)
    df_song_con = pd.DataFrame(song_con_rows)

    excel_path = os.path.join(root, "bang_so_sanh_nse_tong_hop_xgb.xlsx")
    try:
        with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
            df_main.to_excel(writer, sheet_name="Tong_Hop_Phuong_Phap", index=False)
            df_detail.to_excel(writer, sheet_name="Chi_Tiet_Theo_Mua", index=False)
            df_song_con.to_excel(writer, sheet_name="Thu_Nghiem_Song_Con_2", index=False)
            df_horizon.to_excel(writer, sheet_name="Chi_Tiet_Moc_Thoi_Gian", index=False)
    except PermissionError:
        excel_path = os.path.join(root, "bang_so_sanh_nse_tong_hop_xgb_v2.xlsx")
        with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
            df_main.to_excel(writer, sheet_name="Tong_Hop_Phuong_Phap", index=False)
            df_detail.to_excel(writer, sheet_name="Chi_Tiet_Theo_Mua", index=False)
            df_song_con.to_excel(writer, sheet_name="Thu_Nghiem_Song_Con_2", index=False)
            df_horizon.to_excel(writer, sheet_name="Chi_Tiet_Moc_Thoi_Gian", index=False)

    print("\n" + "=" * 90)
    print("BẢNG TỔNG HỢP (CẢ NĂM):")
    print("=" * 90)
    cols = ["Hồ Chứa", "Nhánh Sông", "NSE Single", "NSE Nhánh (pooled, chưa FT)",
            "NSE Lưu vực (pooled, chưa FT)", "NSE Fine-tune từ Nhánh",
            "NSE Fine-tune từ Lưu vực", "Phương Pháp Tốt Nhất"]
    print(df_main[cols].to_string(index=False))
    print(f"\n✅ ĐÃ XUẤT FILE EXCEL: {excel_path}")
    print("=" * 90)
    return excel_path


In [ ]:
generate_summary(root=".")


## Tải kết quả
Toàn bộ thư mục làm việc (`artifacts/`, `eval_json/`, và
`bang_so_sanh_nse_tong_hop_xgb.xlsx`) -- tải về, giải nén đè vào
`XGBoost_Py_Backend/` của project.
